# Untrained Qwen3-4B-Instruct-2507 on the test splits (Milestone 2)

Runs `Qwen/Qwen3-4B-Instruct-2507` (revision `cdbee75f17c01a7cc42f958dc650907174af0554`), untrained, on the test splits under the frozen protocol (`docs/milestone-2/EVAL_PROTOCOL.md`, "Test-split evaluation"): fp16 on one T4, the model's chat template, greedy decoding, `max_new_tokens=128`, prompt version 2. Each declared system runs on each test split once; every session is logged in `docs/milestone-2/TEST_RUN_LOG.md`, including failed ones.

Two sessions (each well inside Kaggle's 12-hour limit):

- **A:** tier 1 `test_in_distribution` (560) and `test_held_out` (960), about 4 hours;
- **B:** tier 2 `test_in_distribution` (256) and `test_held_out` (400), and live test v1 (300), about 3.5 hours.

**Setup:** set `SESSION` below, Accelerator *GPU T4 x2* (only the first GPU is made visible), Internet *on*, attach the private Kaggle Dataset with `hetren-m2-bundle.tar.gz`, then *Save Version → Save & Run All*. The notebook fails if any split does not finish; download `qwen-untrained-test-session-<A|B>.tar.gz` from the Output tab.

In [ ]:
SESSION = "A"  # "A" or "B"

import glob, json, os, shutil, subprocess, sys, tarfile

# One T4 only (protocol): hide the second GPU from every process started below.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

WORK = "/kaggle/working/hetren"
OUT = f"/kaggle/working/qwen-untrained-test-session-{SESSION}"

if not os.path.exists(WORK):
    unpacked = glob.glob("/kaggle/input/**/BUNDLE.json", recursive=True)
    packed = glob.glob("/kaggle/input/**/hetren-m2-bundle.tar.gz", recursive=True)

    if unpacked:  # Kaggle may unpack archives on upload
        shutil.copytree(os.path.dirname(unpacked[0]), WORK)
    else:
        os.makedirs(WORK)
        tarfile.open(packed[0]).extractall(WORK)

print("bundle commit:", json.load(open(f"{WORK}/BUNDLE.json"))["commit"])
os.chdir(WORK)

RUNS = {
    "A": [
        ["--dataset", "artifacts/milestone-2/dataset-v2", "--split", "docs/milestone-2/splits-v1/test_in_distribution.txt"],
        ["--dataset", "artifacts/milestone-2/dataset-v2", "--split", "docs/milestone-2/splits-v1/test_held_out.txt"],
    ],
    "B": [
        ["--dataset", "artifacts/milestone-2/dataset-v3", "--split", "docs/milestone-2/splits-v2/test_in_distribution.txt"],
        ["--dataset", "artifacts/milestone-2/dataset-v3", "--split", "docs/milestone-2/splits-v2/test_held_out.txt"],
        ["--live", "artifacts/milestone-2/live-test-v1"],
    ],
}[SESSION]
SUMMARIES = {
    "A": ["summary-tier1-test_in_distribution.json", "summary-tier1-test_held_out.json"],
    "B": ["summary-tier2-test_in_distribution.json", "summary-tier2-test_held_out.json", "summary-live.json"],
}[SESSION]

In [ ]:
# Pinned versions (the repo's runtime requirements, plus transformers for Qwen3).
!pip install -q jsonschema==4.26.0 openai==3.42.0 openai-agents==0.24.0 tiktoken==0.14.0 transformers==4.56.2 accelerate==1.10.1
!nvidia-smi --query-gpu=index,name,memory.total --format=csv
!python -c "import torch; n = torch.cuda.device_count(); print(n, 'visible GPU:', torch.cuda.get_device_name(0)); assert n == 1"

## Test runs (each split once; a runner that stops fails the notebook)

In [ ]:
for arguments in RUNS:
    subprocess.run([sys.executable, "-m", "packages.support_evaluator.local_model_baseline", *arguments, "--out", OUT],
                   check=True)

In [ ]:
missing = [name for name in SUMMARIES if not os.path.exists(f"{OUT}/{name}")]

with tarfile.open(f"/kaggle/working/qwen-untrained-test-session-{SESSION}.tar.gz", "w:gz") as results:
    results.add(OUT, arcname=os.path.basename(OUT))

print(json.load(open(f"{OUT}/metadata.json")))

if missing:
    raise SystemExit(f"incomplete session {SESSION}: no summary for {missing}")

print(f"session {SESSION} complete: {SUMMARIES}")